# Credit Line Increase Strategy & Risk Analysis
**Author:** Keshav Dhiman

## The business problem
A card issuer increases credit lines in two ways:
- **Proactive Line Increase (PLI):** the bank picks customers and offers them a higher limit.
- **Reactive Line Increase (RLI):** the customer asks for a higher limit and the bank approves or declines it.

A good line-increase strategy gives more limit to customers who will **use** it and **repay** it, and keeps it away from customers who are likely to default.

**Questions this project answers**
1. Which customers use their credit line heavily, and how risky are they? (SQL analysis)
2. How likely is each customer to default? (probability-of-default model)
3. Which customers should get a proactive line increase? (champion vs challenger strategy)
4. How should a customer's request for an increase be decided, with clear reasons? (RLI decision engine)
5. How do we monitor the strategy after launch? (dashboard data and stability checks)

**Data:** UCI *Default of Credit Card Clients*: 30,000 credit card accounts with 6 months of bills, payments and repayment status, plus whether each account defaulted the next month.

## 0. Setup
On **Google Colab**, run the next cell once. Locally, run `pip install duckdb xgboost scikit-learn pandas matplotlib ucimlrepo`.

In [ ]:
# Colab only. Remove the leading '#' to run.
# !pip install -q duckdb xgboost ucimlrepo

In [ ]:
import os
import numpy as np
import pandas as pd
import duckdb                      # lets us run SQL directly on pandas tables
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, roc_curve
from xgboost import XGBClassifier

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.width", 140)

## 1. Load the data
**Option A (recommended):** download `UCI_Credit_Card.csv` from Kaggle (search for "Default of Credit Card Clients Dataset" by UCI Machine Learning) and put it in the repo's `data/` folder (or next to this notebook). On Colab, use the file upload button.

**Option B:** if the CSV isn't found, the cell below downloads the data from the UCI repository automatically.

In [ ]:
# Looks in ../data/ (repo layout) first, then next to the notebook (Colab upload)
DATA_FILE = next((f for f in ["../data/UCI_Credit_Card.csv", "UCI_Credit_Card.csv"] if os.path.exists(f)), "UCI_Credit_Card.csv")

if os.path.exists(DATA_FILE):
    raw = pd.read_csv(DATA_FILE)
else:
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=350)
    raw = pd.concat([ds.data.features, ds.data.targets], axis=1)

# Drop the ID column if present, then give every column a clear name.
# Both sources have the same 23 features + 1 target in the same order.
raw = raw.drop(columns=[c for c in raw.columns if c.upper() == "ID"])
cols = (["limit_amt", "sex", "education", "marriage", "age"]
        + [f"pay_status_{i}" for i in range(1, 7)]    # 1 = most recent month
        + [f"bill_{i}" for i in range(1, 7)]          # statement balance
        + [f"paid_{i}" for i in range(1, 7)]          # amount the customer paid
        + ["default_next_month"])
raw.columns = cols
raw.insert(0, "account_id", np.arange(1, len(raw) + 1))
print(raw.shape)
raw.head()

### What the payment status codes mean
| Code | Meaning |
|---|---|
| -2 | No balance to pay |
| -1 | Paid in full ("transactor") |
| 0 | Paid the minimum and carried the rest ("revolver", the core Pay Over Time customer) |
| 1–9 | Payment delayed by that many months |

## 2. Data quality and control checks
In a regulated business every refresh of the data is checked before anyone uses it. Each check below either passes or is flagged, and the results are kept as a **control log**.

In [ ]:
df = raw.copy()
control_log = []

def log(check, value, ok):
    control_log.append({"check": check, "result": value, "status": "INFO" if ok is None else ("PASS" if ok else "FLAG")})

log("Row count", len(df), len(df) > 0)
log("Missing values (all columns)", int(df.isna().sum().sum()), df.isna().sum().sum() == 0)
log("Duplicate account rows", int(df.drop(columns="account_id").duplicated().sum()),
    df.drop(columns="account_id").duplicated().sum() == 0)
log("Credit limit <= 0", int((df.limit_amt <= 0).sum()), (df.limit_amt <= 0).sum() == 0)
bad_edu = (~df.education.isin([1, 2, 3, 4])).sum()
log("Undocumented education codes (0,5,6)", int(bad_edu), bad_edu == 0)
bad_mar = (~df.marriage.isin([1, 2, 3])).sum()
log("Undocumented marriage codes (0)", int(bad_mar), bad_mar == 0)
over_limit = (df.bill_1 > df.limit_amt).sum()
log("Accounts over limit (bill > limit)", int(over_limit), None)
credit_bal = (df.bill_1 < 0).sum()
log("Accounts with credit (negative) balance", int(credit_bal), None)

# Fix: group the undocumented codes into "Other"
df.loc[~df.education.isin([1, 2, 3, 4]), "education"] = 4
df.loc[~df.marriage.isin([1, 2, 3]), "marriage"] = 3

pd.DataFrame(control_log)

## 3. SQL analysis
DuckDB lets us run SQL straight on the pandas table. This is the same SQL you'd write in BigQuery, Hive or SQL Server.

### 3a. Build account-level features with SQL
- **Utilization** = balance ÷ limit (how much of the line the customer uses)
- **Worst delay** = the worst payment status in the last 6 months
- **Months delayed** = how many of the last 6 months were late
- **Payment ratio** = total paid ÷ total billed over 6 months
- **Customer type** = transactor, revolver or delinquent, based on the latest month

In [ ]:
duckdb.register("raw_accounts", df)

duckdb.sql('''
CREATE OR REPLACE TABLE accounts AS
SELECT
    *,
    bill_1 / NULLIF(limit_amt, 0)                                         AS util_now,
    (bill_1 + bill_2 + bill_3 + bill_4 + bill_5 + bill_6) / 6.0
        / NULLIF(limit_amt, 0)                                            AS util_avg6,
    GREATEST(pay_status_1, pay_status_2, pay_status_3,
             pay_status_4, pay_status_5, pay_status_6)                    AS worst_delay_6m,
    (CAST(pay_status_1 > 0 AS INT) + CAST(pay_status_2 > 0 AS INT) +
     CAST(pay_status_3 > 0 AS INT) + CAST(pay_status_4 > 0 AS INT) +
     CAST(pay_status_5 > 0 AS INT) + CAST(pay_status_6 > 0 AS INT))       AS months_delayed_6m,
    LEAST(COALESCE((paid_1 + paid_2 + paid_3 + paid_4 + paid_5 + paid_6)
        / NULLIF(GREATEST(bill_1 + bill_2 + bill_3 + bill_4 + bill_5 + bill_6, 0), 0), 1), 1)
                                                                          AS pay_ratio_6m,
    CASE WHEN pay_status_1 IN (-2, -1) THEN 'Transactor'
         WHEN pay_status_1 = 0         THEN 'Revolver'
         WHEN pay_status_1 = 1         THEN '1 month late'
         ELSE '2+ months late' END                                        AS customer_type,
    CASE WHEN bill_1 / NULLIF(limit_amt, 0) < 0.10 THEN '1. <10%'
         WHEN bill_1 / NULLIF(limit_amt, 0) < 0.30 THEN '2. 10-30%'
         WHEN bill_1 / NULLIF(limit_amt, 0) < 0.50 THEN '3. 30-50%'
         WHEN bill_1 / NULLIF(limit_amt, 0) < 0.80 THEN '4. 50-80%'
         WHEN bill_1 / NULLIF(limit_amt, 0) <= 1.0 THEN '5. 80-100%'
         ELSE '6. Over limit' END                                         AS util_band
FROM raw_accounts
''')
duckdb.sql("SELECT account_id, limit_amt, util_now, worst_delay_6m, pay_ratio_6m, customer_type FROM accounts LIMIT 5").df()

### 3b. Portfolio overview

In [ ]:
duckdb.sql('''
SELECT COUNT(*)                         AS accounts,
       ROUND(AVG(default_next_month), 4) AS default_rate,
       ROUND(AVG(limit_amt))            AS avg_limit,
       ROUND(AVG(util_now), 3)          AS avg_utilization,
       ROUND(AVG(CASE WHEN customer_type = 'Revolver' THEN 1 ELSE 0 END), 3) AS share_revolvers
FROM accounts
''').df()

### 3c. Default rate by utilization band. Does heavier use of the line mean more risk?

In [ ]:
util_tbl = duckdb.sql('''
SELECT util_band,
       COUNT(*)                                        AS accounts,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio,
       ROUND(AVG(default_next_month), 4)               AS default_rate,
       ROUND(AVG(limit_amt))                           AS avg_limit
FROM accounts
GROUP BY util_band
ORDER BY util_band
''').df()
util_tbl

### 3d. Default rate by customer type

In [ ]:
type_tbl = duckdb.sql('''
SELECT customer_type,
       COUNT(*)                          AS accounts,
       ROUND(AVG(default_next_month), 4) AS default_rate,
       ROUND(AVG(util_now), 3)           AS avg_utilization
FROM accounts
GROUP BY customer_type
ORDER BY default_rate
''').df()
type_tbl

### 3e. Window functions: utilization deciles with running totals
`NTILE(10)` splits customers into 10 equal groups by utilization. `SUM() OVER (ORDER BY …)` builds a running total, which shows what share of all defaults sits in the high-utilization groups.

In [ ]:
duckdb.sql('''
WITH deciles AS (
    SELECT *, NTILE(10) OVER (ORDER BY util_now) AS util_decile
    FROM accounts
),
summary AS (
    SELECT util_decile,
           COUNT(*)                AS accounts,
           MIN(util_now)           AS min_util,
           MAX(util_now)           AS max_util,
           SUM(default_next_month) AS defaults
    FROM deciles
    GROUP BY util_decile
)
SELECT *,
       ROUND(defaults * 1.0 / accounts, 4)                                       AS default_rate,
       ROUND(100.0 * SUM(defaults) OVER (ORDER BY util_decile DESC)
             / SUM(defaults) OVER (), 1)                                         AS cum_pct_defaults_from_top
FROM summary
ORDER BY util_decile
''').df()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(util_tbl.util_band, util_tbl.default_rate * 100, color="#2E5C8A")
ax[0].set_title("Default rate by utilization band"); ax[0].set_ylabel("Default rate (%)")
ax[0].tick_params(axis="x", rotation=30)
ax[1].bar(type_tbl.customer_type, type_tbl.default_rate * 100, color="#2E5C8A")
ax[1].set_title("Default rate by customer type"); ax[1].set_ylabel("Default rate (%)")
plt.tight_layout(); plt.show()

**Write your findings here after running.** For example: *"Default rate rises from X% for accounts under 10% utilization to Y% for accounts over 80%. Customers who were late last month default at Z times the rate of revolvers."*

## 4. Probability-of-default (PD) model
We predict `default_next_month` using **behavioral features only**: how the customer uses and repays the card.

> **Why no sex, age, marriage or education?** Lending decisions are regulated, and using protected or demographic attributes can be unfair or illegal. Keeping the model behavioral is the safer and more defensible choice. Expect an interview question on this.

We compare a simple, explainable model (**Logistic Regression**) against a more powerful one (**XGBoost**).

In [ ]:
acct = duckdb.sql("SELECT * FROM accounts").df()

FEATURES = ["limit_amt", "util_now", "util_avg6", "worst_delay_6m", "months_delayed_6m",
            "pay_ratio_6m", "pay_status_1", "pay_status_2", "bill_1", "paid_1"]
X = acct[FEATURES].fillna(0)
y = acct["default_next_month"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape, " Default rate:", round(y.mean(), 4))

In [ ]:
def ks_stat(y_true, score):
    # KS = max gap between the cumulative % of bads and goods ranked by score.
    fpr, tpr, _ = roc_curve(y_true, score)
    return (tpr - fpr).max()

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logit.fit(X_train, y_train)

xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8, eval_metric="auc", random_state=42)
xgb.fit(X_train, y_train)

results = []
for name, model in [("Logistic Regression", logit), ("XGBoost", xgb)]:
    p_tr = model.predict_proba(X_train)[:, 1]
    p_te = model.predict_proba(X_test)[:, 1]
    results.append({"model": name,
                    "train_AUC": roc_auc_score(y_train, p_tr),
                    "test_AUC": roc_auc_score(y_test, p_te),
                    "test_KS": ks_stat(y_test, p_te),
                    "test_Gini": 2 * roc_auc_score(y_test, p_te) - 1})
results = pd.DataFrame(results)
results

**How to read this:** AUC is the chance the model ranks a random defaulter as riskier than a random non-defaulter (0.5 = coin flip, 1.0 = perfect). KS is the maximum separation between defaulters and non-defaulters. A big gap between train and test AUC would mean overfitting.

In [ ]:
best_name = results.sort_values("test_AUC", ascending=False).iloc[0]["model"]
best = xgb if best_name == "XGBoost" else logit
print("Best model:", best_name)

test = acct.loc[X_test.index].copy()
test["pd"] = best.predict_proba(X_test)[:, 1]
train_scores = best.predict_proba(X_train)[:, 1]

# Risk bands: 10 equal groups by PD (1 = safest, 10 = riskiest)
test["risk_band"] = pd.qcut(test["pd"].rank(method="first"), 10, labels=range(1, 11)).astype(int)
band_tbl = (test.groupby("risk_band")
                .agg(accounts=("pd", "size"), avg_pd=("pd", "mean"),
                     actual_default_rate=("default_next_month", "mean"),
                     defaults=("default_next_month", "sum")))
band_tbl["cum_pct_defaults_captured"] = (band_tbl["defaults"][::-1].cumsum()[::-1]
                                         / band_tbl["defaults"].sum() * 100)
band_tbl

**Check:** the actual default rate should rise steadily from band 1 to band 10. That's what makes the model usable for decisions.

In [ ]:
imp = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values()
coef = pd.Series(logit[-1].coef_[0], index=FEATURES).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
imp.plot.barh(ax=ax[0], color="#2E5C8A", title="XGBoost feature importance")
coef.plot.barh(ax=ax[1], color=["#B23A48" if v > 0 else "#2E5C8A" for v in coef],
               title="Logistic coefficients (red = raises risk)")
plt.tight_layout(); plt.show()

## 5. Proactive Line Increase strategy: champion vs challenger
- **Champion** = the current strategy. Here it's a typical rule-based policy: *no late payment in 6 months and utilization of at least 50%.*
- **Challenger** = the proposed strategy: the same safety gates, plus the **PD model** instead of a hard payment rule, aimed at customers who use the line (utilization ≥ 30%).

Both run on the **test set** (accounts the model never saw), so the comparison is honest.

**Policy gates (always applied):** not currently past due, no 2+ month delay in the last 6 months, and not over limit.

**Line increase offered:** 20% of the current limit, rounded to the nearest 1,000 and capped at 50,000.

In [ ]:
gates = (test.pay_status_1 <= 0) & (test.worst_delay_6m < 2) & (test.util_now <= 1.0)
test["increase_amt"] = np.minimum((test.limit_amt * 0.20).round(-3), 50_000)

def evaluate(mask, name):
    s = test[mask]
    exp_new_balance = (s.increase_amt * s.util_now.clip(0, 1)).sum()   # assume they use the new line like the old
    exp_loss = (s.pd * s.increase_amt * s.util_now.clip(0, 1)).sum()   # PD x extra exposure (LGD assumed 100%)
    return {"strategy": name, "eligible_accounts": int(mask.sum()),
            "pct_of_book": mask.mean() * 100,
            "actual_default_rate": s.default_next_month.mean(),
            "expected_new_balance": exp_new_balance,
            "expected_loss": exp_loss,
            "loss_to_balance_pct": exp_loss / exp_new_balance * 100 if exp_new_balance else np.nan}

champion = gates & (test.worst_delay_6m <= 0) & (test.util_now >= 0.5)
champ_res = evaluate(champion, "Champion (rules)")

# Sweep PD cut-offs: find the most accounts we can approve while keeping risk <= champion
sweep = []
for cut in np.arange(0.05, 0.41, 0.01):
    m = gates & (test.util_now >= 0.3) & (test.pd <= cut)
    if m.sum() > 0:
        sweep.append({"pd_cutoff": round(cut, 2), "eligible": int(m.sum()),
                      "default_rate": test.loc[m, "default_next_month"].mean()})
sweep = pd.DataFrame(sweep)
ok = sweep[sweep.default_rate <= champ_res["actual_default_rate"]]
PD_CUTOFF = ok.sort_values("eligible").iloc[-1]["pd_cutoff"] if len(ok) else sweep.iloc[0]["pd_cutoff"]
print("Chosen PD cut-off:", PD_CUTOFF)

challenger = gates & (test.util_now >= 0.3) & (test.pd <= PD_CUTOFF)
chall_res = evaluate(challenger, f"Challenger (PD <= {PD_CUTOFF})")
strategy = pd.DataFrame([champ_res, chall_res])
strategy

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sweep.eligible, sweep.default_rate * 100, marker="o", ms=3, color="#2E5C8A", label="Challenger at each PD cut-off")
ax.scatter(champ_res["eligible_accounts"], champ_res["actual_default_rate"] * 100, color="#B23A48", s=80, zorder=3, label="Champion")
ax.set_xlabel("Accounts eligible for line increase"); ax.set_ylabel("Actual default rate (%)")
ax.set_title("Volume vs risk trade-off"); ax.legend(); plt.tight_layout(); plt.show()

lift = (chall_res["eligible_accounts"] / champ_res["eligible_accounts"] - 1) * 100
print(f"Challenger approves {lift:+.1f}% accounts vs champion, at a default rate of "
      f"{chall_res['actual_default_rate']*100:.2f}% vs {champ_res['actual_default_rate']*100:.2f}%.")

### Swap set analysis
Which customers does the challenger **add** (swap-in) and **remove** (swap-out) compared to the champion? Risk teams always ask this.

In [ ]:
swap = pd.DataFrame({
    "group": ["Both approve", "Swap-in (challenger only)", "Swap-out (champion only)"],
    "accounts": [int((champion & challenger).sum()), int((~champion & challenger).sum()), int((champion & ~challenger).sum())],
    "default_rate": [test.loc[champion & challenger, "default_next_month"].mean(),
                     test.loc[~champion & challenger, "default_next_month"].mean(),
                     test.loc[champion & ~challenger, "default_next_month"].mean()]})
swap

## 6. Reactive Line Increase: decision engine with reason codes
When a customer **asks** for an increase, the bank must decide, and if it declines, it must give the **reason** (adverse-action rules require this in the US). Every decision here returns reason codes so it can be explained and audited.

In [ ]:
def rli_decision(a, pd_cutoff=PD_CUTOFF):
    reasons = []
    if a.pay_status_1 > 0:      reasons.append("R01 Account currently past due")
    if a.worst_delay_6m >= 2:   reasons.append("R02 Serious delinquency in last 6 months")
    if a.util_now > 1.0:        reasons.append("R03 Account over credit limit")
    if a.pd > pd_cutoff:        reasons.append("R04 Risk score above approval threshold")
    if a.pay_ratio_6m < 0.05:   reasons.append("R05 Low repayment relative to balance")
    if reasons:
        return pd.Series({"decision": "DECLINE", "increase": 0, "reasons": "; ".join(reasons)})
    return pd.Series({"decision": "APPROVE", "increase": a.increase_amt, "reasons": "-"})

test[["rli_decision", "rli_increase", "rli_reasons"]] = test.apply(rli_decision, axis=1).values
test[["account_id", "limit_amt", "util_now", "pd", "rli_decision", "rli_increase", "rli_reasons"]].head(8)

In [ ]:
print("RLI approval rate:", round((test.rli_decision == "APPROVE").mean() * 100, 1), "%")
reason_counts = (test.loc[test.rli_decision == "DECLINE", "rli_reasons"]
                     .str.split("; ").explode().value_counts())
reason_counts.to_frame("declines_citing_reason")

## 7. Monitoring: score stability (PSI)
After a strategy goes live, we check that the customers being scored still look like the ones the model was built on. **PSI (Population Stability Index)** compares the two score distributions: under 0.10 is stable, 0.10–0.25 needs watching, over 0.25 is a significant shift.

In [ ]:
def psi(expected, actual, bins=10):
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected)
    a = np.histogram(actual, edges)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return np.sum((a - e) * np.log(a / e))

score_psi = psi(train_scores, test["pd"])
print(f"PSI (train vs test scores): {score_psi:.4f} ->",
      "Stable" if score_psi < 0.1 else "Monitor" if score_psi < 0.25 else "Significant shift")

## 8. Export data for the Power BI dashboard
Two files to load into Power BI:
- `pli_account_level.csv`: one row per account, with segment, risk band and decisions
- `pli_segment_summary.csv`: pre-aggregated KPIs by segment

In [ ]:
export = test[["account_id", "limit_amt", "util_now", "util_band", "customer_type", "worst_delay_6m",
               "pd", "risk_band", "increase_amt", "rli_decision", "rli_reasons", "default_next_month"]].copy()
export["champion_eligible"] = champion.astype(int).values
export["challenger_eligible"] = challenger.astype(int).values
export.to_csv("pli_account_level.csv", index=False)

seg = (export.groupby(["util_band", "customer_type"])
             .agg(accounts=("account_id", "count"), default_rate=("default_next_month", "mean"),
                  avg_pd=("pd", "mean"), champion_eligible=("champion_eligible", "sum"),
                  challenger_eligible=("challenger_eligible", "sum"))
             .reset_index())
seg.to_csv("pli_segment_summary.csv", index=False)
pd.DataFrame(control_log).to_csv("control_log.csv", index=False)
print("Saved: pli_account_level.csv, pli_segment_summary.csv, control_log.csv")

## 9. Your results: copy these into your CV

In [ ]:
b = results.set_index("model").loc[best_name]
print(f"- Best model: {best_name} | test AUC {b.test_AUC:.2f} | KS {b.test_KS*100:.0f} | Gini {b.test_Gini:.2f}")
print(f"- Challenger vs champion: {lift:+.0f}% eligible accounts "
      f"({chall_res['eligible_accounts']:,} vs {champ_res['eligible_accounts']:,}), "
      f"default rate {chall_res['actual_default_rate']*100:.2f}% vs {champ_res['actual_default_rate']*100:.2f}%")
print(f"- Expected new balance: {chall_res['expected_new_balance']:,.0f} vs {champ_res['expected_new_balance']:,.0f}")
print(f"- RLI approval rate: {(test.rli_decision=='APPROVE').mean()*100:.1f}% | PSI: {score_psi:.3f}")

## 10. Limitations (say these in the interview, because it shows maturity)
- "Default next month" is a short-horizon target. Real line-increase models predict risk over 12+ months.
- There's no income, months-on-book or bureau data, so the line-increase amount is a simple rule.
- Expected loss assumes 100% loss given default and that customers use the new line like the old one.
- A real launch would run the challenger as an **A/B test** on a small random group before rolling it out.